# Directed residual GNNs (SIGN + APPNP)


This notebook produces the main later standalone graph results from raw data:

- **Directed SIGN residual** (best validation-selected standalone GNN)
- **Directed APPNP residual** (strongest standalone GNN on test context)

In [ ]:
from pathlib import Path
import json, os, random

import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss
import torch
import torch.nn as nn
import torch.nn.functional as F

PROJECT_DIR = Path('..')
RAW_DIR = PROJECT_DIR / 'Dataset' / 'raw' / 'elliptic_bitcoin_dataset'

OUT_DIR = PROJECT_DIR / 'outputs' / 'directed_residual_gnns'
OUT_DIR.mkdir(parents=True, exist_ok=True)

ACTIVE_SPLIT = {'train': [1, 32], 'val': [33, 37], 'test': [38, 42]}
SEEDS = [7, 17, 27, 37, 47, 57]
TRAIN_START = 13
ANCHOR_CACHE = OUT_DIR / 'newsplit_32_37_42_et600_anchor_probs.npy'
OUT_SEED_CSV = OUT_DIR / 'sign_appnp_raw_reproduced_seed_results.csv'
OUT_SUMMARY = OUT_DIR / 'sign_appnp_raw_reproduced_summary.json'

os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except Exception:
    pass

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MAX_EPOCHS = 40
RECENCY_GAMMA = 0.95
WEIGHT_DECAY = 1e-4
DROPOUT_LINEAR = 0.2
LR_LINEAR = 0.01
APPNP_ALPHA = 0.2
APPNP_K = 4
EPS = 1e-12


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class LinearResidual(nn.Module):
    def __init__(self, d_in: int, dropout: float = 0.2):
        super().__init__()
        self.drop = nn.Dropout(dropout)
        self.linear = nn.Linear(d_in, 1)
        self.scale_param = nn.Parameter(torch.tensor(0.0))
    def forward(self, x_concat: torch.Tensor, anchor_logit: torch.Tensor) -> torch.Tensor:
        raw_delta = self.linear(self.drop(x_concat)).squeeze(-1)
        scale = 2.0 * torch.sigmoid(self.scale_param)
        delta = torch.tanh(raw_delta) * scale
        return anchor_logit + delta


In [2]:
# Load data
feat = pd.read_csv(RAW_DIR / 'elliptic_txs_features.csv', header=None)
feat.columns = ['txId', 'time_step'] + [f'f{i}' for i in range(165)]

classes = pd.read_csv(RAW_DIR / 'elliptic_txs_classes.csv')
df = feat.merge(classes, on='txId', how='left')
df['y'] = pd.to_numeric(df['class'], errors='coerce').map({1: 1, 2: 0})

X_local = df[[f'f{i}' for i in range(93)]].to_numpy(np.float32)
X_all = df[[f'f{i}' for i in range(165)]].to_numpy(np.float32)
y = df['y'].to_numpy()
t = df['time_step'].to_numpy(np.int16)
labeled = ~np.isnan(y)
n = len(df)

idx_map = pd.Series(np.arange(n, dtype=np.int32), index=df['txId'].values)
edges = pd.read_csv(RAW_DIR / 'elliptic_txs_edgelist.csv')
src = idx_map.loc[edges['txId1']].to_numpy(np.int32)
dst = idx_map.loc[edges['txId2']].to_numpy(np.int32)

print("loaded rows:", n)
print("mapped edges:", len(src))
print("labeled counts:",
      int((labeled & (t >= 1) & (t <= 32)).sum()),
      int((labeled & (t >= 33) & (t <= 37)).sum()),
      int((labeled & (t >= 38) & (t <= 42)).sum()))

loaded rows: 203769
mapped edges: 234355
labeled counts: 28938 4503 6436


In [3]:
# Stage-1 ET-600 anchor
if ANCHOR_CACHE.exists():
    p_anchor = np.load(ANCHOR_CACHE)
else:
    p_anchor = np.full(n, np.nan, dtype=np.float32)
    for cur_t in range(9, 33):
        fit_mask = labeled & (t >= cur_t - 8) & (t <= cur_t - 1)
        pred_mask = (t == cur_t)
        model = ExtraTreesClassifier(
            n_estimators=600,
            max_features='sqrt',
            class_weight='balanced_subsample',
            random_state=7,
            n_jobs=-1,
        )
        model.fit(X_all[fit_mask], y[fit_mask].astype(int))
        p_anchor[pred_mask] = model.predict_proba(X_all[pred_mask])[:, 1].astype(np.float32)

    full_train_mask = labeled & (t >= 1) & (t <= 32)
    model_full = ExtraTreesClassifier(
        n_estimators=600,
        max_features='sqrt',
        class_weight='balanced_subsample',
        random_state=7,
        n_jobs=-1,
    )
    model_full.fit(X_all[full_train_mask], y[full_train_mask].astype(int))
    pred_mask = (t >= 33) & (t <= 42)
    p_anchor[pred_mask] = model_full.predict_proba(X_all[pred_mask])[:, 1].astype(np.float32)
    np.save(ANCHOR_CACHE, p_anchor)

need_mask = (t >= TRAIN_START) & (t <= 42)
missing = int(np.isnan(p_anchor[need_mask]).sum())
if missing:
    raise RuntimeError(f'Missing anchor predictions: {missing}')

In [4]:
# Restrict to the exact relevant time window
subset_mask = (t >= TRAIN_START) & (t <= 42)
sub_idx = np.flatnonzero(subset_mask)
sub_n = len(sub_idx)
global_to_sub = np.full(n, -1, dtype=np.int32)
global_to_sub[sub_idx] = np.arange(sub_n, dtype=np.int32)

edge_mask = subset_mask[src] & subset_mask[dst]
sub_src = global_to_sub[src[edge_mask]]
sub_dst = global_to_sub[dst[edge_mask]]

X_local_sub = X_local[sub_idx]
y_sub = y[sub_idx]
t_sub = t[sub_idx]
labeled_sub = labeled[sub_idx]

p_sub = np.clip(p_anchor[sub_idx], 1e-6, 1 - 1e-6)
logit_sub = np.log(p_sub / (1 - p_sub)).astype(np.float32)
abslogit_sub = np.abs(logit_sub).astype(np.float32)

x_base = np.concatenate([
    X_local_sub,
    p_sub[:, None],
    logit_sub[:, None],
    abslogit_sub[:, None],
], axis=1).astype(np.float32)

sub_in_deg = np.bincount(sub_dst, minlength=sub_n).astype(np.float32)
sub_out_deg = np.bincount(sub_src, minlength=sub_n).astype(np.float32)
sub_tot_deg = sub_in_deg + sub_out_deg

win = 1.0 / np.maximum(sub_in_deg[sub_dst], 1.0)
wout = 1.0 / np.maximum(sub_out_deg[sub_src], 1.0)
rows_all = np.concatenate([sub_src, sub_dst])
cols_all = np.concatenate([sub_dst, sub_src])
wall = 1.0 / np.maximum(sub_tot_deg[rows_all], 1.0)

A_in = sparse.csr_matrix((win, (sub_dst, sub_src)), shape=(sub_n, sub_n), dtype=np.float32)
A_out = sparse.csr_matrix((wout, (sub_src, sub_dst)), shape=(sub_n, sub_n), dtype=np.float32)
A_all = sparse.csr_matrix((wall, (rows_all, cols_all)), shape=(sub_n, sub_n), dtype=np.float32)

std_mask = (t_sub >= TRAIN_START) & (t_sub <= 32)
train_mask = labeled_sub & (t_sub >= TRAIN_START) & (t_sub <= 32)
val_mask = labeled_sub & (t_sub >= 33) & (t_sub <= 37)
test_mask = labeled_sub & (t_sub >= 38) & (t_sub <= 42)

train_y_np = y_sub[train_mask].astype(np.float32)
val_y_np = y_sub[val_mask].astype(np.float32)
test_y_np = y_sub[test_mask].astype(np.float32)
train_steps_np = t_sub[train_mask].astype(np.int64)

train_anchor_logit = torch.from_numpy(logit_sub[train_mask]).to(DEVICE)
val_anchor_logit = torch.from_numpy(logit_sub[val_mask]).to(DEVICE)
test_anchor_logit = torch.from_numpy(logit_sub[test_mask]).to(DEVICE)

train_y = torch.from_numpy(train_y_np).to(DEVICE)
recency_weights_np = (RECENCY_GAMMA ** (32 - train_steps_np)).astype(np.float32)
recency_weights_np = recency_weights_np / recency_weights_np.mean()
recency_weights = torch.from_numpy(recency_weights_np).to(DEVICE)
pos_weight = torch.tensor((train_y_np == 0).sum() / max((train_y_np == 1).sum(), 1), dtype=torch.float32, device=DEVICE)

def standardize_view(v_full: np.ndarray):
    mu = v_full[std_mask].mean(axis=0, dtype=np.float64).astype(np.float32)
    sd = v_full[std_mask].std(axis=0, dtype=np.float64).astype(np.float32)
    sd = np.where(sd < 1e-5, 1.0, sd).astype(np.float32)
    vv = ((v_full - mu) / sd).astype(np.float32)
    return vv[train_mask], vv[val_mask], vv[test_mask]

def build_sign10_mats():
    mats_train, mats_val, mats_test = [], [], []
    x0 = x_base
    tr, va, te = standardize_view(x0)
    mats_train.append(tr); mats_val.append(va); mats_test.append(te)

    in1 = (A_in @ x0).astype(np.float32)
    out1 = (A_out @ x0).astype(np.float32)
    all1 = (A_all @ x0).astype(np.float32)
    for v in [in1, out1, all1]:
        tr, va, te = standardize_view(v)
        mats_train.append(tr); mats_val.append(va); mats_test.append(te)

    in2 = (A_in @ in1).astype(np.float32)
    out2 = (A_out @ out1).astype(np.float32)
    all2 = (A_all @ all1).astype(np.float32)
    for v in [in2, out2, all2]:
        tr, va, te = standardize_view(v)
        mats_train.append(tr); mats_val.append(va); mats_test.append(te)

    in3 = (A_in @ in2).astype(np.float32)
    out3 = (A_out @ out2).astype(np.float32)
    all3 = (A_all @ all2).astype(np.float32)
    for v in [in3, out3, all3]:
        tr, va, te = standardize_view(v)
        mats_train.append(tr); mats_val.append(va); mats_test.append(te)

    return (
        np.concatenate(mats_train, axis=1).astype(np.float32),
        np.concatenate(mats_val, axis=1).astype(np.float32),
        np.concatenate(mats_test, axis=1).astype(np.float32),
    )

def build_appnp4_mats(alpha: float = 0.2, K: int = 4):
    def prop(A, x):
        h = x.copy()
        for _ in range(K):
            h = ((1.0 - alpha) * (A @ h) + alpha * x).astype(np.float32)
        return h
    mats_train, mats_val, mats_test = [], [], []
    for v in [x_base, prop(A_in, x_base), prop(A_out, x_base), prop(A_all, x_base)]:
        tr, va, te = standardize_view(v)
        mats_train.append(tr); mats_val.append(va); mats_test.append(te)
    return (
        np.concatenate(mats_train, axis=1).astype(np.float32),
        np.concatenate(mats_val, axis=1).astype(np.float32),
        np.concatenate(mats_test, axis=1).astype(np.float32),
    )

In [5]:
def fit_family(name: str, Xtr_np: np.ndarray, Xva_np: np.ndarray, Xte_np: np.ndarray, seeds: list[int], lr: float, dropout: float):
    Xtr = torch.from_numpy(Xtr_np).to(DEVICE)
    Xva = torch.from_numpy(Xva_np).to(DEVICE)
    Xte = torch.from_numpy(Xte_np).to(DEVICE)
    rows = []
    for seed in seeds:
        set_seed(seed)
        model = LinearResidual(d_in=Xtr.shape[1], dropout=dropout).to(DEVICE)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
        best = {'val_pr': -1.0, 'epoch': -1, 'test_prob': None}
        for epoch in range(1, MAX_EPOCHS + 1):
            model.train()
            opt.zero_grad(set_to_none=True)
            train_logits = model(Xtr, train_anchor_logit)
            loss_vec = F.binary_cross_entropy_with_logits(train_logits, train_y, reduction='none', pos_weight=pos_weight)
            loss = (loss_vec * recency_weights).mean()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            opt.step()

            model.eval()
            with torch.no_grad():
                val_prob = torch.sigmoid(model(Xva, val_anchor_logit)).cpu().numpy()
                test_prob = torch.sigmoid(model(Xte, test_anchor_logit)).cpu().numpy()
            val_pr = float(average_precision_score(val_y_np, val_prob))
            if val_pr > best['val_pr']:
                best = {'val_pr': val_pr, 'epoch': epoch, 'test_prob': test_prob.copy()}
        rows.append({
            'family': name,
            'seed': seed,
            'best_epoch': int(best['epoch']),
            'val_pr_auc': float(best['val_pr']),
            'test_pr_auc': float(average_precision_score(test_y_np, best['test_prob'])),
            'test_roc_auc': float(roc_auc_score(test_y_np, best['test_prob'])),
            'test_brier': float(brier_score_loss(test_y_np, best['test_prob'])),
        })
    return pd.DataFrame(rows).sort_values(['family', 'val_pr_auc', 'seed'], ascending=[True, False, True]).reset_index(drop=True)

In [6]:
sign_Xtr, sign_Xva, sign_Xte = build_sign10_mats()
appnp_Xtr, appnp_Xva, appnp_Xte = build_appnp4_mats(alpha=APPNP_ALPHA, K=APPNP_K)

sign_seed_df = fit_family('sign_linear10_start13_local_rerun', sign_Xtr, sign_Xva, sign_Xte, SEEDS, lr=LR_LINEAR, dropout=DROPOUT_LINEAR)
appnp_seed_df = fit_family('appnp_linear4_start13_local_rerun', appnp_Xtr, appnp_Xva, appnp_Xte, SEEDS, lr=LR_LINEAR, dropout=DROPOUT_LINEAR)

seed_df = pd.concat([sign_seed_df, appnp_seed_df], ignore_index=True)
seed_df.to_csv(OUT_SEED_CSV, index=False)

sign_best_val = sign_seed_df.iloc[0].to_dict()
appnp_best_val = appnp_seed_df.iloc[0].to_dict()
appnp_best_test = appnp_seed_df.sort_values(['test_pr_auc', 'val_pr_auc'], ascending=[False, False]).iloc[0].to_dict()

summary = {
    'split': ACTIVE_SPLIT,
    'sign_best_validation_selected': sign_best_val,
    'appnp_best_validation_selected': appnp_best_val,
    'appnp_strongest_test_context': appnp_best_test,
}
with open(OUT_SUMMARY, 'w') as f:
    json.dump(summary, f, indent=2)

print('SIGN best validation-selected test PR-AUC:', sign_best_val['test_pr_auc'])
print('APPNP strongest test-context PR-AUC:', appnp_best_test['test_pr_auc'])

SIGN best validation-selected test PR-AUC: 0.9153551698745114
APPNP strongest test-context PR-AUC: 0.9150206856114556
